In [ ]:
from util import *
from QEM.trainer import QuantumGraphTrainer
from QEM.preprocessor import DataPreProcessor

In [ ]:
filename = '../data/processed_data_graph_q10_d10.csv'
data_type = "graph"  # Change to "freq" for frequency conversion.
n_qubits = 4

processor = DataPreProcessor(filename, file_format="csv", data_type=data_type, n_qubits=n_qubits)
converted_data = processor.convert_data()

## OPTUNA

In [ ]:
def objective(trial):
    """Optimization target function for quantum-graph neural network tuning"""
    params = {
        'hidden_channels': trial.suggest_int("hidden_channels", 16, 256),
        'mlp_hidden': trial.suggest_int("mlp_hidden", 8, 128),
        'batch_size': trial.suggest_int("batch_size", 16, 256),
        'lr': trial.suggest_float("lr", 1e-5, 1e-1, log=True),
        'epochs': trial.suggest_int("epochs", 50, 500),
        'mult_noisy': trial.suggest_float("mult_noisy", 0.1, 2.0),
        'mult_pauli': trial.suggest_float("mult_pauli", 0.1, 2.0),
        'mult_mlp': trial.suggest_float("mult_mlp", 0.1, 2.0),
        'mult_graph': trial.suggest_float("mult_graph", 0.1, 2.0)
    }

    trainer = QuantumGraphTrainer(
        converted_data,
        n_qubits=n_qubits,
        test_size=0.2,
        **params
    )
    
    trainer.train()
    pred, targ = trainer.evaluate()
    return np.sqrt(mean_squared_error(targ.flatten(), pred.flatten()))

# Create named study with optimization settings
study = optuna.create_study(
    study_name="Hybrid GNN Tuner",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(objective, n_trials=50, gc_after_trial=True)

# Save optimization results
study.trials_dataframe().to_csv("hybrid_gnn_tuning_results_q4_d10.csv")

print(f"Best trial for {study.study_name}:")
print(f"RMSE: {study.best_value:.4f}")
print("Optimal parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


## WANDB

In [ ]:
import wandb
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Inside the train() function, after computing errors:
def train():
    wandb.init()
    config = wandb.config

    # Set the run name based on hyperparameters
    run_name = (
        f"hc={config.hidden_channels}_mlph={config.mlp_hidden}_bs={config.batch_size}_"
        f"lr={config.lr}_epochs={config.epochs}_"
        f"m_noisy={config.mult_noisy}_m_pauli={config.mult_pauli}_"
        f"m_mlp={config.mult_mlp}_m_graph={config.mult_graph}"
    )
    wandb.run.name = run_name

    trainer = QuantumGraphTrainer(
        converted_data,    # ensure converted_data is defined elsewhere
        n_qubits=n_qubits, # ensure n_qubits is defined
        test_size=0.2,
        **config
    )

    trainer.train()
    pred, targ = trainer.evaluate()

    y = targ.flatten()
    y_hat = pred.flatten()

    # Compute errors
    mse_score = mean_squared_error(y, y_hat)
    mae_score = mean_absolute_error(y, y_hat)
    rmse_score = np.sqrt(mse_score)

    mse_error = np.square(y - y_hat)
    rmse_error = np.sqrt(mse_error)
    mae_error = np.abs(y - y_hat)

    # Plot MSE Error Distribution
    plt.figure(figsize=(8, 6))
    plt.hist(mse_error, bins=50, color='blue', alpha=0.7)
    plt.title("MSE Error Distribution")
    plt.xlabel("Error")
    plt.ylabel("Frequency")
    mse_plot_path = "mse_error_plot.png"
    plt.savefig(mse_plot_path)
    wandb.log({"MSE Error Distribution": wandb.Image(mse_plot_path)})
    
    # Plot RMSE Error Distribution
    plt.figure(figsize=(8, 6))
    plt.hist(rmse_error, bins=50, color='green', alpha=0.7)
    plt.title("RMSE Error Distribution")
    plt.xlabel("Error")
    plt.ylabel("Frequency")
    rmse_plot_path = "rmse_error_plot.png"
    plt.savefig(rmse_plot_path)
    wandb.log({"RMSE Error Distribution": wandb.Image(rmse_plot_path)})

    # Plot MAE Error Distribution
    plt.figure(figsize=(8, 6))
    plt.hist(mae_error, bins=50, color='orange', alpha=0.7)
    plt.title("MAE Error Distribution")
    plt.xlabel("Error")
    plt.ylabel("Frequency")
    mae_plot_path = "mae_error_plot.png"
    plt.savefig(mae_plot_path)
    wandb.log({"MAE Error Distribution": wandb.Image(mae_plot_path)})

    # Log metrics and errors
    wandb.log({
        "MSE": mse_score,
        "RMSE": rmse_score,
        "MAE": mae_score,
        "Max_MSE_Error": np.max(mse_error),
        "Mean_MSE_Error": np.mean(mse_error),
        "Max_MAE_Error": np.max(mae_error),
        "Mean_MAE_Error": np.mean(mae_error),
        "Max_RMSE_Error": np.max(rmse_error),
        "Mean_RMSE_Error": np.mean(rmse_error),
        "Example_Errors": wandb.Table(
            columns=["Prediction", "Target", "MSE_Error", "MAE_Error"],
            data=list(zip(y_hat[:100], y[:100], mse_error[:100], mae_error[:100]))
        )
    })

    wandb.finish()



# Sweep configuration for hyperparameter tuning.
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'RMSE', 'goal': 'minimize'},
    'parameters': {
        "hidden_channels": {"values": [32, 64, 128]},
        "mlp_hidden": {"values": [16, 32, 64]},
        "batch_size": {"values": [32, 64, 128]},
        "lr": {"values": [0.001, 0.01, 0.1]},
        "epochs": {"value": 100},
        "mult_noisy": {"values": [0.5, 1, 1.5]},
        "mult_pauli": {"values": [0.5, 1, 1.5]},
        "mult_mlp": {"values": [0.5, 1, 1.5]},
        "mult_graph": {"values": [0.5, 1, 1.5]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="QuantumGraphTuning")
wandb.agent(sweep_id, function=train)
